# Laboratorio 06 — Redis: datos en memoria y streaming

**Procesamiento Masivo de Datos (ELE051-B / EIN102B)** · USM 2026-2 · Paralelo 701
**Clase 7** · Viernes 25 de septiembre de 2026 · Lab. de Informática 2

---

Ejercicios para trabajar en clase. No se entrega ni se evalúa: el notebook queda para ustedes.

Primer laboratorio del bloque de **streaming**. Hasta ahora los datos *ya estaban ahí* (un shapefile, un extracto de OSM). Hoy llegan de a uno, ~2 por segundo, desde 16 sensores del Gran Concepción, y no paran. Tres pasos: Redis como motor clave-valor (Parte 1), Pub/Sub — la versión ingenua, que pierde lo que no escuchó (Parte 2) — y Streams — la versión durable, que lo guarda y lo reparte (Parte 3). El material de estudio está en `CONTENTS.md`, en esta carpeta.

## Cómo trabajar este notebook

- En equipos de 2–3 personas, un notebook por equipo.
- **Todo el notebook es Python**, con la librería `redis-py` (`r.set(...)`, `r.hset(...)`, `r.xadd(...)`). Cada ejercicio indica el comando equivalente de Redis "puro" y la sección de `CONTENTS.md` que lo explica: si quieren verlo en consola, pruébenlo en `redis-cli` (CONTENTS.md §2.2). La traducción es casi mecánica: el comando en minúsculas es el método (`SET` → `r.set`, `ZREVRANGE` → `r.zrevrange`).
- Donde vean `___` hay que completar. Cada celda trae una **pista** con el método a usar; la firma completa está en la [documentación de redis-py](https://redis.readthedocs.io/en/stable/commands.html).
- **Fuente de datos:** un Redis compartido en un servidor del curso (el *VPS*), al que todo el curso se conecta a la vez. La clave se da en clase.
- **Camino de rescate:** si el VPS no conecta, la Parte 0 levanta un Redis propio con `docker compose` y le reproduce `datos/gran_concepcion_stream.parquet` (los mismos datos, con la misma lógica). Todo el notebook funciona igual en ambos caminos; solo cambia la escala de la Parte 3.
- **Plan B del profesor:** si el VPS no está, el profesor puede levantar el mismo servidor en un PC de la sala. Cuando dé su IP, cambien `VPS_HOST` y `VPS_PORT` en el `.env`, reinicien el kernel y ejecuten desde el principio (si venían del camino de rescate, primero la celda de Limpieza).
- Hay celdas que **esperan a propósito** (60 s en el 1.1, 20 s en el 2.2, unos segundos en la Parte 3): no las interrumpan.

## Configuración

Antes de ejecutarla: copien `.env.example` como `.env` y completen ahí `EQUIPO` y `VPS_PASS` (la clave se da en clase). El `.env` **no se sube a git** — es donde viven las claves, fuera del código. Después ejecuten la celda una vez.

In [ ]:
import json
import os
import subprocess
import threading
import time
from collections import defaultdict, deque
from datetime import datetime

import pandas as pd
import redis

# La conexión vive en el archivo .env de esta carpeta (no va a git: tiene la clave). Ver .env.example.
if os.path.exists(".env"):
    for linea in open(".env", encoding="utf-8"):
        k, _, v = linea.strip().partition("=")
        if k and not k.startswith("#"):
            os.environ[k.strip()] = v.strip()

EQUIPO = os.environ.get("EQUIPO", "equipo-1")    # ← su número de equipo (en el .env): equipo-1, equipo-2, ...
VPS_HOST = os.environ.get("VPS_HOST", "redis-stream.5.75.154.51.sslip.io")
VPS_PORT = int(os.environ.get("VPS_PORT", 443))  # 443 = HTTPS, con TLS · Plan B del profesor: 6379, sin TLS
VPS_PASS = os.environ.get("VPS_PASS", "")        # la clave se da en clase

P = f"{EQUIPO}:"                                 # prefijo de las claves propias (ver Parte 1)
STREAM = "sensores:flujo"
CANAL = "sensores:flujo_vehicular"
GRUPO = "curso-pmd"
PARQUET = "datos/gran_concepcion_stream.parquet"


def hora(ms):
    """ID de stream (o milisegundos) → hora legible."""
    return datetime.fromtimestamp(int(str(ms).split("-")[0]) / 1000).strftime("%H:%M:%S")


# --- Verificación del entorno ---
print("✅ redis-py", redis.__version__)
print(("✅ .env · " + EQUIPO + " → " + VPS_HOST + ":" + str(VPS_PORT)) if os.path.exists(".env") else "⚠️  falta .env — copien .env.example como .env y completen la clave que se da en clase")
print(("✅ " if os.path.exists(PARQUET) else "⚠️  falta ") + PARQUET + " (solo se usa en el camino de rescate — ver README.md)")
d = subprocess.run(["docker", "compose", "version"], capture_output=True, text=True)
print(("✅ " + d.stdout.strip()) if d.returncode == 0 else "⚠️  `docker compose` no responde (solo se usa en el camino de rescate)")

---
# Parte 0 — Levantar y verificar la fuente de datos

16 estaciones sintéticas del Gran Concepción (la tabla completa está en la guía y en `generador.py`), tres tipos de sensor. Cada evento trae cuatro campos:

| Campo | Tipo | Ejemplo |
|---|---|---|
| `estacion_id` | texto | `CC-001` |
| `tipo_sensor` | texto | `flujo_vehicular` · `ocupacion_estacionamiento` · `calidad_aire` |
| `valor` | número | flujo: veh/min (5–120) · ocupación: % (0–100) · aire: PM2.5 µg/m³ (5–80) |
| `timestamp` | ISO 8601 | momento en que el sensor midió |

Cada evento se escribe **dos veces**: con `PUBLISH` en el canal `sensores:flujo_vehicular` (Parte 2) y con `XADD` en el stream `sensores:flujo` (Parte 3). Una fracción pequeña llega **tarde** o **repetida** — a propósito.

Esta celda intenta el VPS; si no conecta en 3 segundos, levanta el Redis local (`docker-compose.yml` de esta carpeta). `r` es **la conexión**: todo lo que sigue es `r.<comando>(...)`.

In [ ]:
def conectar():
    try:
        r = redis.Redis(host=VPS_HOST, port=VPS_PORT, password=VPS_PASS or None, ssl=VPS_PORT == 443,
                        socket_connect_timeout=3, decode_responses=True)
        r.ping()
        print("✓ Conectado al stream compartido del VPS —", VPS_HOST)
        return r, "vps"
    except Exception as e:
        print("✗ Sin conexión al VPS (", e, ") — levantando camino de rescate local")
        subprocess.run(["docker", "compose", "up", "-d", "--wait"], check=True)
        r = redis.Redis(host="127.0.0.1", port=6379, decode_responses=True)
        r.ping()
        return r, "local"


r, origen = conectar()

Si `origen == "local"`, la siguiente celda arranca la reproducción del parquet hacia el Redis local, en un hilo aparte, al mismo ritmo que el VPS (~2 eventos/s). Con el VPS no hace nada: allá el generador ya está corriendo.

In [ ]:
parar_reproduccion = threading.Event()


def reproducir_parquet(r, ruta=PARQUET, ritmo=0.5):
    df = pd.read_parquet(ruta)          # en orden de LLEGADA: no ordenar por timestamp (las tardías son parte del ejercicio)
    for fila in df.to_dict("records"):
        if parar_reproduccion.is_set():
            break
        r.publish(CANAL, json.dumps(fila))
        r.xadd(STREAM, fila)
        time.sleep(ritmo)


if origen == "local":
    threading.Thread(target=reproducir_parquet, args=(r,), daemon=True).start()
    time.sleep(3)
print(origen, "· eventos en el stream:", r.xlen(STREAM))
r.xrevrange(STREAM, count=3)        # en consola: XREVRANGE sensores:flujo + - COUNT 3

**Esperado:** un mensaje de conexión (`vps` o `local`) y las 3 últimas entradas del stream, cada una como `(id, {campos})`. Fíjense en el formato del ID (`1790...-0`): lo usaremos en la Parte 3.

---
# Parte 1 — Redis como motor clave-valor

Cinco estructuras, cinco preguntas distintas (CONTENTS.md §3). Una advertencia de convivencia: si están en el VPS, **todo el curso comparte el mismo Redis**, y un `SET estacion:CC-001 …` de un equipo pisa el del otro. Por eso las claves de esta parte llevan el prefijo del equipo, `P` (→ `equipo-1:`). Es la práctica real: en un Redis compartido, el *namespace* va en el nombre de la clave, porque Redis no tiene esquemas ni tablas que separen.

### Ejercicio 1.1 — Strings y TTL

Guarden `42.5` como último valor de CC-001, con un tiempo de vida de 60 segundos, y léanlo de vuelta junto con el tiempo que le queda.

En consola (§3.1–3.2): `SET <clave> 42.5 EX 60` · `GET <clave>` · `TTL <clave>`

In [ ]:
clave = f"{P}estacion:CC-001:ultimo_valor"
___                                  # Pista: r.set(clave, valor, ex=segundos)
t_set = time.time()
print("GET →", ___)                  # Pista: r.get(...)
print("TTL →", ___, "s")             # Pista: r.ttl(...)

Ahora se espera a que venza (la celda duerme lo que falte para completar 60 s):

In [ ]:
time.sleep(max(0, 61 - (time.time() - t_set)))
print("GET →", r.get(clave))
print("TTL →", r.ttl(clave), " (-2 = la clave no existe)")

**Esperado:** el segundo `GET` devuelve `None` (`nil` en `redis-cli`) — la clave se autodestruyó sola.

**✍️ Respuesta del equipo — 1.1:** ¿qué otro dato del curso, hasta ahora, se hubiera beneficiado de un TTL? `___`

### Ejercicio 1.2 — Hash: la ficha de una estación

Guarden la ficha de CC-001 (nombre, comuna, tipo — tabla de la guía) como un hash, y léanla completa y solo la comuna.

En consola (§3.3): `HSET <clave> nombre "Plaza Independencia" comuna "Concepción" tipo "flujo_vehicular"` · `HGETALL <clave>` · `HGET <clave> comuna`

In [ ]:
___        # Pista: r.hset(clave, mapping={"nombre": ..., "comuna": ..., "tipo": ...})
print(___)  # Pista: r.hgetall(clave) → un dict
print(___)  # Pista: r.hget(clave, campo)

Repitan para 2–3 estaciones más de la tabla, y muestren las fichas con un `for`:

In [ ]:
fichas = {
    "TAL-001": {___},
    # ...
}
# Pista: un for que haga r.hset(...) por cada ficha, y otro que las imprima con r.hgetall(...)

### Ejercicio 1.3 — Listas: bitácora acotada

Empujen las lecturas `38`, `41` y `42.5` (en ese orden) a la bitácora de CC-001, lean las 10 más recientes y recorten la lista a 100 elementos.

En consola (§3.4): `LPUSH <clave> 38 41 42.5` · `LRANGE <clave> 0 9` · `LTRIM <clave> 0 99`

In [ ]:
log = f"{P}estacion:CC-001:log"
___        # Pista: r.lpush(clave, v1, v2, v3)
print(___)  # Pista: r.lrange(clave, inicio, fin)
___        # Pista: r.ltrim(clave, inicio, fin)

**✍️ Respuesta del equipo — 1.3:**

1. ¿En qué orden quedaron los tres valores, y por qué? `___`
2. ¿Por qué `LTRIM` importa en un sistema que corre semanas, no minutos? `___`

### Ejercicio 1.4 — Sets: quién está activo

Dos minutos de reportes: en el actual reportaron `CC-001 TAL-001 HUA-002 PEN-001`; en el anterior, `CC-001 TAL-001 HUA-002 COR-001`. ¿Quiénes reportaron en ambos? ¿Quién dejó de reportar?

En consola (§3.5): `SADD <clave> CC-001 TAL-001 ...` · `SINTER <clave1> <clave2>` · `SDIFF <clave1> <clave2>`

In [ ]:
actual, anterior = f"{P}activos:min_actual", f"{P}activos:min_anterior"
___   # Pista: r.sadd(clave, "CC-001", "TAL-001", ...)
___
print("En ambos minutos  (SINTER):", ___)   # Pista: r.sinter(a, b)
print("Dejó de reportar  (SDIFF): ", ___)   # Pista: r.sdiff(a, b) — ¿en qué orden van a y b?

**Esperado:** `SDIFF` devuelve `{'COR-001'}` — dejó de reportar entre un minuto y otro. El orden del `SINTER` puede variar: un set no tiene orden.

Ahora con datos reales: ¿qué estaciones reportaron en el último minuto del stream? La lectura del stream está dada (`xrange` se ve en la Parte 3); armen el set.

In [ ]:
ultimo_min = r.xrange(STREAM, min=int(time.time() * 1000) - 60_000)   # lista de (id, {campos}) del último minuto
activas = ___   # Pista: un set con el campo "estacion_id" de cada entrada
___             # Pista: r.sadd(clave, *activas)
print(len(ultimo_min), "eventos en el último minuto ·", ___, "estaciones distintas")   # Pista: r.scard(...)

### Ejercicio 1.5 — Sorted sets: ranking en vivo

Armen un ranking de congestión (`CC-001`: 45, `HUA-002`: 78, `TAL-001`: 30, `SPP-002`: 92), muestren el top 5, súmenle 15 a CC-001 y vuelvan a mostrarlo.

En consola (§3.6): `ZADD <clave> 45 CC-001 78 HUA-002 ...` · `ZREVRANGE <clave> 0 4 WITHSCORES` · `ZINCRBY <clave> 15 CC-001`

In [ ]:
ranking = f"{P}ranking:congestion"
___         # Pista: r.zadd(clave, {"CC-001": 45, ...})   ← ojo: miembro → puntaje
print(___)  # Pista: r.zrevrange(clave, 0, 4, withscores=True)
___         # Pista: r.zincrby(clave, cantidad, miembro)
print(___)

Y el ranking **real**: el último valor de flujo de cada estación, según las últimas 500 entradas del stream. `ZADD` sobre una estación que ya existe reemplaza su puntaje — no hay que borrar ni reordenar nada.

In [ ]:
for _, ev in r.xrevrange(STREAM, count=500)[::-1]:   # de la más vieja a la más nueva
    if ev["tipo_sensor"] == "flujo_vehicular":
        ___   # Pista: r.zadd(...) con {estacion: valor}; el valor llega como texto → float(...)
___           # el top 5

**✍️ Preguntas de cierre de la Parte 1:**

1. ¿Cuál de las cinco estructuras usarían para "la última lectura de cada estación, visible en un dashboard"? ¿Por qué? `___`
2. ¿Cuál para "las 5 estaciones más congestionadas ahora mismo"? `___`

---
# Parte 2 — Pub/Sub: la versión ingenua

### Ejercicio 2.1 — Publicar y suscribirse

Primero la mecánica, en un canal propio (`{P}prueba`, para no mezclar sus mensajes de prueba con los del curso): suscribirse, publicar un mensaje y leerlo.

En consola (§4), en dos terminales: `SUBSCRIBE <canal>` en una, `PUBLISH <canal> '{"estacion":"CC-001","valor":42.5}'` en la otra.

In [ ]:
canal_prueba = f"{P}prueba"
ps = r.pubsub()
___                                              # Pista: ps.subscribe(canal)
print("suscripción:", ps.get_message(timeout=1))
n = ___                                          # Pista: r.publish(canal, texto) — el mensaje, con json.dumps({...})
print("PUBLISH lo recibieron", n, "suscriptor(es)")
print("mensaje:", ps.get_message(timeout=1))
ps.close()

`publish` devuelve **cuántos** suscriptores lo recibieron — y nada más. Redis no lo guarda en ninguna parte.

Ahora un suscriptor de verdad, al canal del curso, que corre en un hilo aparte y agrega en vivo: el **promedio móvil** de los últimos `N` valores de cada estación. El hilo ya está armado; lo que falta es `procesar()`, que se llama con cada evento recibido (un dict con los cuatro campos).

In [ ]:
class Suscriptor:
    def __init__(self, r, canal=CANAL, n=10):
        self.r, self.canal = r, canal
        self.recibidos = 0
        self.ventanas = defaultdict(lambda: deque(maxlen=n))   # estacion_id → últimos n valores
        self._parar = threading.Event()

    def procesar(self, ev):
        # TODO: agregar el valor del evento a la ventana de su estación.
        # Pista: self.ventanas[...] es un deque: .append(...). El valor puede llegar como texto → float(...)
        ___

    def _correr(self):
        ps = self.r.pubsub(ignore_subscribe_messages=True)
        ps.subscribe(self.canal)
        while not self._parar.is_set():
            m = ps.get_message(timeout=0.5)
            if m:
                self.recibidos += 1
                try:
                    self.procesar(json.loads(m["data"]))
                except (ValueError, KeyError, TypeError):
                    pass            # mensaje ajeno o mal formado (en el VPS, cualquiera puede publicar)
        ps.close()

    def iniciar(self):
        self._parar.clear()
        self.hilo = threading.Thread(target=self._correr, daemon=True)
        self.hilo.start()
        return self

    def detener(self):
        self._parar.set()
        self.hilo.join()

    def promedios(self):
        # TODO: una pd.Series estación → promedio de su ventana (redondeado a 1 decimal)
        # Pista: {e: sum(v) / len(v) for e, v in self.ventanas.items() if v}
        ___

In [ ]:
sus = Suscriptor(r).iniciar()
time.sleep(1)                     # dar tiempo a que la suscripción quede activa
n0 = sus.recibidos
time.sleep(10)
frecuencia = (sus.recibidos - n0) / 10
print(f"{sus.recibidos - n0} mensajes en 10 s → {frecuencia:.1f} eventos/s")
sus.promedios()

**Esperado:** ~20 mensajes en 10 s (~2 eventos/s) y el promedio móvil de las estaciones que alcanzaron a reportar. Vuelvan a ejecutar `sus.promedios()` en un rato: cambia solo, porque el hilo sigue escuchando.

### Ejercicio 2.2 — El experimento: apagar el suscriptor

La celda anota la hora exacta, detiene el suscriptor, espera 20 segundos (el generador sigue publicando) y lo reinicia. Guarda `t_ini` y `t_fin` en milisegundos: la Parte 3 los va a necesitar.

In [ ]:
sus.detener()
t_ini = int(time.time() * 1000)
print("⏸  suscriptor detenido a las", hora(t_ini), f"(t_ini = {t_ini})")
recibidos_antes = sus.recibidos
time.sleep(20)
t_fin = int(time.time() * 1000)
sus.iniciar()
print("▶️  suscriptor reiniciado a las", hora(t_fin), f"(t_fin = {t_fin})")
time.sleep(3)
print("Mensajes recibidos después de reiniciar:", sus.recibidos - recibidos_antes,
      "— solo los nuevos; de los 20 s de pausa, ninguno")

No hay manera de pedirle a Redis "lo que me perdí": el canal no tiene historial. ¿Cuántos eventos se perdieron?

In [ ]:
perdidos_estimados = ___   # Pista: frecuencia (eventos/s, del 2.1) × duración de la pausa (t_ini y t_fin están en ms)
print("Eventos perdidos con Pub/Sub (estimado):", perdidos_estimados)

**✍️ Respuesta del equipo — 2.2:** ¿en qué parte de un sistema real esto sería aceptable, y en cuál no? `___`

---
# Parte 3 — Streams: la versión durable

### Ejercicio 3.1 — Recuperar exactamente lo perdido

`XADD` corrió en paralelo desde la Parte 0: cada evento que se publicó en el canal también quedó en el stream. Este ejercicio solo **lee** lo que ya estaba guardado, entre `t_ini` y `t_fin` — los IDs de un stream **son** milisegundos, así que se puede pedir un rango de tiempo directo.

En consola (§5.2): `XRANGE sensores:flujo <t_ini> <t_fin>`

In [ ]:
recuperados = ___   # Pista: r.xrange(STREAM, min=..., max=...)
print(len(recuperados), "entradas entre", hora(t_ini), "y", hora(t_fin))
recuperados[:3]

In [ ]:
print(f"Pub/Sub: se perdieron ~{perdidos_estimados} eventos · Streams: se recuperaron {len(recuperados)} para el mismo intervalo")

**Esperado:** aparecen las entradas de los 20 segundos del 2.2, completas. El conteo debería acercarse a lo que estimaron.

Dos detalles de los datos (CONTENTS.md §5.5). La tabla ya calcula `atraso_s`: cuántos segundos pasaron entre que el sensor midió (`timestamp`) y que la entrada llegó a Redis (el ID). Cuenten las repetidas y las tardías.

In [ ]:
df = pd.DataFrame([{"id": i, **c} for i, c in recuperados])
df["llegada"] = pd.to_datetime(df["id"].str.split("-").str[0].astype("int64"), unit="ms", utc=True)
df["medicion"] = pd.to_datetime(df["timestamp"], format="ISO8601", utc=True)
df["atraso_s"] = (df["llegada"] - df["medicion"]).dt.total_seconds().round(1)
print("Repetidas (mismo evento dos veces):", ___)         # Pista: df.duplicated([columnas]).sum()
print("Tardías (medidas > 10 s antes de llegar):", ___)   # Pista: (condición sobre df["atraso_s"]).sum()
df[["id", "estacion_id", "valor", "timestamp", "atraso_s"]].sort_values("atraso_s", ascending=False).head()

**✍️ Respuesta del equipo — 3.1:**

1. Una línea: "eventos perdidos con Pub/Sub" vs. "eventos recuperados con Streams", mismo intervalo. `___`
2. El ID de la entrada dice *cuándo llegó a Redis*; el campo `timestamp` dice *cuándo midió el sensor*. ¿Cuál usarían para "el promedio de flujo entre 09:00 y 09:05"? ¿Por qué? `___`
3. ¿Qué hacen con las repetidas? `___`

### Ejercicio 3.2 — Leer en vivo

Pidan al stream **solo lo nuevo desde ahora** (`$`), esperando hasta 5 segundos a que llegue algo.

En consola (§5.3): `XREAD BLOCK 5000 STREAMS sensores:flujo $`

In [ ]:
___   # Pista: r.xread({STREAM: "$"}, block=milisegundos)

**✍️ Respuesta del equipo — 3.2:** ¿qué pasa si dos equipos hacen exactamente esto al mismo tiempo, contra el mismo stream del VPS? `___`

### Ejercicio 3.3 — Consumer groups: repartir el trabajo

Todo el curso usa **el mismo grupo** (`curso-pmd`); cada equipo es un consumidor distinto dentro de él (`EQUIPO`). El primer equipo en llegar crea el grupo; los demás reciben `BUSYGROUP` — está bien, significa que ya existe.

En consola (§5.4): `XGROUP CREATE sensores:flujo curso-pmd $`

In [ ]:
try:
    ___   # Pista: r.xgroup_create(STREAM, GRUPO, id="$")
    print("Grupo creado")
except redis.ResponseError as e:
    print("El grupo ya existía →", e)

Pidan un lote de hasta 5 entradas **nuevas para el grupo** (`>`), a nombre de su equipo.

En consola: `XREADGROUP GROUP curso-pmd <equipo> COUNT 5 STREAMS sensores:flujo >`

In [ ]:
time.sleep(4)   # un grupo recién creado con $ parte vacío: se deja llegar un lote
lote = ___      # Pista: r.xreadgroup(GRUPO, consumidor, {STREAM: ">"}, count=5, block=5000)
ids = [i for _, entradas in lote for i, _ in entradas]
print(EQUIPO, "recibió", ids)

Confirmen (`XACK`) solo **las tres primeras**, como si el equipo se hubiera caído a mitad del lote, y miren qué quedó pendiente — de ustedes y del resto del curso.

En consola: `XACK sensores:flujo curso-pmd <id1> <id2> <id3>` · `XPENDING sensores:flujo curso-pmd`

In [ ]:
print("XACK →", ___, "confirmadas")   # Pista: r.xack(STREAM, GRUPO, *ids_a_confirmar)
resumen = ___                           # Pista: r.xpending(STREAM, GRUPO) → un dict
print(f"Pendientes en el grupo: {resumen['pending']}")
pd.DataFrame(resumen["consumers"])

Ahora el reparto, **dentro del equipo**: tres consumidores (`<equipo>-a`, `-b`, `-c`) piden un lote cada uno. Si están en el camino de rescate, esta es su versión del ejercicio — la mecánica es idéntica, solo cambia la escala.

In [ ]:
time.sleep(8)   # que se acumulen entradas para los tres
lotes = {}
for c in ["a", "b", "c"]:
    lote = ___   # Pista: lo mismo de arriba, con el consumidor f"{EQUIPO}-{c}"
    lotes[c] = {i for _, entradas in lote for i, _ in entradas}
    print(f"{EQUIPO}-{c}:", sorted(lotes[c]))
print("¿Alguna entrada repetida entre consumidores?", ___)   # Pista: intersección de sets: lotes["a"] & lotes["b"] ...

**Esperado:** consumidores distintos reciben **lotes distintos** — ninguna entrada se repite.

**✍️ Respuesta del equipo — 3.3:** ¿en qué se diferencia esto del `XREAD` del 3.2? `___`

### Ejercicio 3.4 — Reclamar una entrada abandonada (si sobra tiempo)

Las 2 entradas sin `XACK` del 3.3 siguen pendientes a nombre de `EQUIPO`, y las de `-a`, `-b`, `-c` también. `XCLAIM` le "roba" una entrada a otro consumidor, pero solo si lleva más de 30 s sin confirmarse. En el VPS, busquen una de **otro equipo**. La búsqueda está dada; el reclamo, no.

En consola: `XCLAIM sensores:flujo curso-pmd <equipo> 30000 <id>`

In [ ]:
pend = r.xpending_range(STREAM, GRUPO, min="-", max="+", count=100)
ajenas = [p for p in pend if p["consumer"] != EQUIPO and p["time_since_delivered"] >= 30_000]
if not ajenas:
    print("Nada abandonado hace más de 30 s todavía — esperen un poco y vuelvan a ejecutar.")
else:
    victima = ajenas[0]
    print("Reclamando", victima["message_id"], "de", victima["consumer"], f"(inactiva {victima['time_since_delivered'] / 1000:.0f} s)")
    print(___)   # Pista: r.xclaim(STREAM, GRUPO, nuevo_consumidor, min_idle_time=30_000, message_ids=[...])
    print("Ahora es de:", r.xpending_range(STREAM, GRUPO, min=victima["message_id"], max=victima["message_id"], count=1)[0]["consumer"])

**✍️ Respuesta del equipo — 3.4:** ¿por qué un sistema real necesita poder "robarle" trabajo pendiente a un consumidor que se cayó? `___`

---
# Parte 4 — Síntesis del equipo

Tres líneas:

1. Cuántos eventos se perdieron con Pub/Sub vs. cuántos se recuperaron con Streams, para el mismo intervalo. `___`
2. ¿Conectaron al VPS o usaron el camino de rescate? ¿Notaron diferencia de comportamiento? `___`
3. Una estructura de Redis (de las cinco vistas) que usarían en su **proyecto semestral**, y para qué dato específico. `___`

---
### Limpieza (antes de irse)

Detiene el suscriptor y la reproducción. Si usaron el camino de rescate, `docker compose down` borra el contenedor `pmd-redis` — y con él todo lo que tenía: no hay volumen.

In [ ]:
sus.detener()
parar_reproduccion.set()
if origen == "local":
    subprocess.run(["docker", "compose", "down"], check=True)
print("Listo.")

---
*Fin del Laboratorio 06. Próxima clase: **HBase** — si Redis resuelve la velocidad (la última hora, en microsegundos), HBase resuelve el volumen (los últimos cinco años, repartidos en disco).*